##  第一阶段：启动与配置（Shell & Hydra）
- 负责拉起分布式集群，分配GPU资源

<b>Ray 负责分布式调度，Hydra 管理复杂配置，FSDP/Megatron 处理模型并行，模块化设计支持灵活扩展。</b>

In [ ]:
# 查看集群上所有作业
ray job list --address="http://..."

# 查看特定作业日志
ray job logs <job_id> --address="http://..."

# 停止作业
ray job stop <job_id> --address="http://..."

# 提交并跟踪日志（-f 表示 follow）
ray job submit --address="..." --follow -- python3 -m recipe.dapo.main_dapo

### 场景：提交 DAPO 训练作业到 Ray 集群

In [ ]:
# 完整命令
ray job submit \
    --address="http://192.168.1.100:8265" \
    --working-dir="/home/user/dapo_project" \
    --runtime-env-json='{
        "pip": ["hydra-core", "omegaconf", "torch", "ray[train]"],
        "env_vars": {"CUDA_VISIBLE_DEVICES": "0,1,2,3"}
    }' \
    -- \
    python3 -m recipe.dapo.main_dapo \
        training.batch_size=256 \
        optimizer.lr=0.0001 \
        model.name=llama2-7b \
        --multirun  # Hydra 批量实验

### python 入口加载 （recipe/dapo/main_dapo.py）

1. python解释器加载 main_dapo.py
2. 使用@hydra.main()装饰器读取config/dapo_trainer.yaml中的配置
3. 覆盖配置：将shell脚本传入的命令行参数合并到配置对象cfg中
4. 调用：执行run_ppo(cfg)函数
   - 初始化ray：ray.init()
   - TaskRunner，首先调用@ray.remote(num_cpus=1),将TaskRunner类转换为ray actor，并申请一个CPU，用于调度（head节点负责集群管理和调度，不运行计算任务，真正的计算在worker中进行）

#### OmegaConf.resolve(config)作用

In [ ]:
# 配置文件中可能有变量引用
paths:
  base: /data/models
  checkpoint: ${paths.base}/llama2-7b  # 未解析时是字符串

# resolve=True 后变成：
paths:
  base: /data/models
  checkpoint: /data/models/llama2-7b   # 实际值

#### 模型下载与 Tokenizer 初始化

copy_to_local 的作用：
- 集群节点可能无法直接访问 HDFS/S3
- 自动将模型权重下载到本地 SSD，加速后续读取

In [ ]:
# 从 HDFS/远程存储下载到本地
local_path = copy_to_local(config.actor_rollout_ref.model.path)

# 初始化 HuggingFace Tokenizer 和 Processor
from verl.utils import hf_processor, hf_tokenizer

tokenizer = hf_tokenizer(local_path)
processor = hf_processor(local_path, use_fast=True)  # 多模态 LLM 使用

#### 分布式策略选择（FSDP vs Megatron）

| 策略             | 适用场景          | 特点                |
| -------------- | ------------- | ----------------- |
| **FSDP/FSDP2** | 中小规模模型（< 70B） | PyTorch 原生，易于调试   |
| **Megatron**   | 大规模模型（> 70B）  | NVIDIA 优化，张量并行效率高 |


In [ ]:
from verl.single_controller.ray import RayWorkerGroup

if config.actor_rollout_ref.actor.strategy in {"fsdp", "fsdp2"}:
    assert config.critic.strategy in {"fsdp", "fsdp2"}
    # 加载fsdp组件
    from verl.workers.fsdp_workers import ActorRolloutRefWorker, CriticWorker
    ray_worker_group_cls = RayWorkerGroup

elif config.actor_rollout_ref.actor.strategy == "megatron":
    assert config.actor_rollout_ref.actor.strategy == config.critic.strategy
    # 加载megatron组件
    from verl.workers.megatron_workers import ActorRolloutRefWorker, CriticWorker
    ray_worker_group_cls = RayWorkerGroup

#### 5. 角色与资源映射（核心设计）

In [ ]:
from verl.trainer.ppo.ray_trainer import ResourcePoolManager, Role

# 定义角色到 Worker 类的映射
role_worker_mapping = {
    Role.ActorRollout: ray.remote(ActorRolloutRefWorker),  # 策略网络 + 采样
    Role.Critic: ray.remote(CriticWorker),                  # 价值网络
}

# 资源池规格：每个节点 n_gpus_per_node 个 GPU，共 nnodes 个节点
global_pool_id = "global_pool"
resource_pool_spec = {
    global_pool_id: [config.trainer.n_gpus_per_node] * config.trainer.nnodes,
    # 例如 2 节点 × 8 GPU = [8, 8]
}

# 角色分配到哪个资源池
mapping = {
    Role.ActorRollout: global_pool_id,
    Role.Critic: global_pool_id,
}

In [ ]:
# 资源池可视化
Resource Pool "global_pool": [8, 8]  # 2 节点，每节点 8 GPU

Role.ActorRollout  ──┐
Role.Critic        ──┼──► 共享这 16 个 GPU
Role.RewardModel   ──┘   (通过 Ray 调度动态分配)

### 可选组件动态加载
#### Reward Model（可选）

In [ ]:
if config.reward_model.enable:
    if config.reward_model.strategy in {"fsdp", "fsdp2"}:
        from verl.workers.fsdp_workers import RewardModelWorker
    elif config.reward_model.strategy == "megatron":
        from verl.workers.megatron_workers import RewardModelWorker
    
    # 动态添加到角色映射
    role_worker_mapping[Role.RewardModel] = ray.remote(RewardModelWorker)
    mapping[Role.RewardModel] = global_pool_id

#### Reference Policy（用于 KL 散度）

条件加载逻辑：
- 只有启用 KL 惩罚时才需要 Reference Policy
- 避免不必要的内存和计算开销

In [ ]:
if config.algorithm.use_kl_in_reward or config.actor_rollout_ref.actor.use_kl_loss:
    # 复用 ActorRolloutRefWorker 作为参考策略
    role_worker_mapping[Role.RefPolicy] = ray.remote(ActorRolloutRefWorker)
    mapping[Role.RefPolicy] = global_pool_id

### 奖励函数管理器（多源奖励）

| 类型              | 用途 | 组成                         |
| --------------- | -- | -------------------------- |
| `reward_type=0` | 训练 | 模型奖励（RM 打分）+ 规则奖励（格式/长度检查） |
| `reward_type=1` | 验证 | 纯规则奖励（更稳定，避免 RM 过拟合）       |


In [ ]:
reward_fn = load_reward_manager(
    config,
    tokenizer,
    reward_type=0,  # 训练用：模型奖励 + 规则奖励组合
    max_resp_len=config.data.max_response_length,
    overlong_buffer_cfg=config.reward_model.overlong_buffer,
)

# 验证专用（纯规则-based，更稳定）
val_reward_fn = load_reward_manager(
    config,
    tokenizer,
    reward_type=1,  # 验证用：仅规则奖励
    max_resp_len=config.data.max_response_length,
    overlong_buffer_cfg=config.reward_model.overlong_buffer,
)

###  资源池管理与训练器初始化

In [1]:
resource_pool_manager = ResourcePoolManager(
    resource_pool_spec=resource_pool_spec,  # [8, 8]
    mapping=mapping                           # 角色→池映射
)

trainer = RayDAPOTrainer(
    config=config,
    tokenizer=tokenizer,
    processor=processor,
    role_worker_mapping=role_worker_mapping,      # 角色类定义
    resource_pool_manager=resource_pool_manager,    # GPU 资源分配
    ray_worker_group_cls=ray_worker_group_cls,      # RayWorkerGroup
    reward_fn=reward_fn,                            # 训练奖励
    val_reward_fn=val_reward_fn,                      # 验证奖励
)

NameError: name 'ResourcePoolManager' is not defined

### 启动训练

init_workers() 内部动作：
1. 根据 resource_pool_spec 在 Ray 集群上申请 GPU 资源
2. 在每个 GPU 上实例化对应的 Worker Actor
3. 建立 Worker 之间的通信组（NCCL 集合通信）
4. 加载模型权重到各 Worker

In [ ]:
trainer.init_workers()  # 初始化所有 Ray Worker（实际创建 Actor）
trainer.fit()           # 开始 DAPO 训练循环

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│                    DAPO Training Loop                       │
├─────────────────────────────────────────────────────────────┤
│  1. ActorRollout.generate()                                   │
│     └── 生成 responses（rollout）                           │
│         └── 使用当前策略 π_θ 采样动作                        │
│                                                               │
│  2. RewardModel.score() / RuleReward.compute()               │
│     └── 计算奖励 r_t = r_model + r_rule - β·KL(π_θ‖π_ref)   │
│                                                               │
│  3. Critic.estimate_value()                                   │
│     └── 计算状态价值 V(s_t)                                  │
│                                                               │
│  4. 计算优势函数 A(s_t, a_t) = GAE(returns, values)          │
│                                                               │
│  5. Actor.update_policy()                                     │
│     └── 最大化 PPO-clip 目标：L^CLIP(θ)                       │
│                                                               │
│  6. Critic.update_value()                                     │
│     └── 最小化价值损失：MSE(V(s_t), returns)                 │
│                                                               │
│  7. 重复直到收敛                                              │
└─────────────────────────────────────────────────────────────┘

# 第二阶段： 分布式环境初始化（ray & manager）